## 1. 조건부 확률

우리가 일상에서 친구와 대화할 때를 떠올려 봅시다. 친구가 "나는 오늘 점심으로 맛있는..." 이라고 말하다가 멈췄다면, 우리는 자연스럽게 뒤에 **피자를** 혹은 **햄버거를** 이라는 단어가 올 것이라고 예상합니다. 왜 **책상을** 이라는 단어는 전혀 예상하지 않을까요?

우리의 뇌는 과거 수십 년 동안 들었던 수많은 한국어 문장 데이터를 바탕으로, "맛있는" 이라는 특정 조건이 주어졌을 때 뒤에 "음식"이 등장한 횟수를 무의식적으로 계산하고 있기 때문입니다. 

만약 우리가 100권의 소설책 데이터를 컴퓨터로 분석했다고 가정해 봅시다. 
소설책 전체에서 **맛있는** 이라는 단어가 총 10번 등장했습니다. 
그리고 그 10번 중에서 **맛있는** 바로 뒤에 **피자를** 이 등장한 횟수가 7번이었습니다.
그렇다면 우리는 아주 단순한 나눗셈을 통해, **맛있는** 다음 **피자를** 이 올 확률이 10번 중 7번, 즉 70%라고 결론을 내릴 수 있습니다. 이것이 통계적 언어 모델이 확률을 계산하는 가장 원초적이고 직관적인 방식입니다.

## 1. 확률의 연쇄 법칙과 언어 모델의 본질

앞선 직관적 계산법을 바탕으로 이제 정식 알고리즘 수식을 도입해 보겠습니다.
언어 모델이 문장 전체의 타당성을 평가하는 수학적 원리는 결합 확률(Joint Probability)에 근거합니다. 단어 시퀀스 $W = w_1, w_2, ..., w_n$ 이 주어졌을 때, 이 문장이 나타날 전체 확률 $P(W)$는 연쇄 법칙(Chain Rule)에 의해 다음과 같이 분해됩니다.

$P(w_1, w_2, ..., w_n) = P(w_1) \times P(w_2 | w_1) \times \dots \times P(w_n | w_1, ..., w_{n-1})$

하지만 이 수식은 완벽함에도 불구하고 치명적인 연산의 한계를 지닙니다. 문장이 길어질수록 조건부에 해당하는 과거 단어의 나열이 무한정 길어지며, 학습 말뭉치에서 이와 완벽하게 동일한 긴 패턴을 찾을 확률은 사실상 0에 수렴하기 때문입니다.

## 2. 마르코프 가정(Markov Assumption)의 도입

이러한 연산의 마비를 해결하기 위해 러시아 수학자 마르코프의 이론을 차용합니다. 직전의 단어 몇 개만 잘라서 확률을 계산하겠다는 실무적 타협안입니다. 이를 N-gram에 적용하면 수식이 비약적으로 단순화됩니다.

- 1-gram (Unigram): 이전 문맥을 완전히 무시합니다. $P(w_n)$
- 2-gram (Bigram): 직전 1개 단어만 의존합니다. $P(w_n | w_{n-1})$
- 3-gram (Trigram): 직전 2개 단어만 의존합니다. $P(w_n | w_{n-2}, w_{n-1})$

## 3. 최대 우도 추정(MLE) 기반의 확률 계산과 증명

통계적 모델에서 확률을 구하는 정석은 말뭉치에 등장한 빈도(Count)를 활용한 최대 우도 추정(MLE)입니다. Bigram 연산식은 다음과 같습니다.

$P(w_n | w_{n-1}) = \frac{Count(w_{n-1}, w_n)}{Count(w_{n-1})}$

**실제 데이터 수학적 증명**
- 말뭉치 1: `<s>` 나는 오늘 파이토치 학습을 한다 `</s>`
- 말뭉치 2: `<s>` 나는 내일 자연어 처리를 한다 `</s>`

위 말뭉치에서 **나는** 다음에 **오늘** 이 올 확률을 MLE 방식으로 계산합니다.
분모 $Count(나는)$을 탐색합니다. 말뭉치 1과 2에서 총 2회 등장했습니다.
분자 $Count(나는, 오늘)$을 탐색합니다. 말뭉치 1에서만 1회 등장했습니다.
결과적으로 $P(오늘 | 나는) = \frac{1}{2} = 0.5$입니다. 50%의 수학적 확률로 다음 단어를 타당하게 예측해 내며 알고리즘 연산 원리가 완벽히 증명됩니다.

## 1. N-gram 확률 연산 엔진 설계
단순 카운팅을 넘어, 말뭉치를 동적으로 입력받아 분자와 분모 데이터베이스를 자동 구축하는 객체 지향 클래스를 설계합니다.

In [1]:
from collections import Counter
class MLELanguateModel:
    def __init__(self, n):
        self.n = n
        self.ngram_counts = Counter()
        self.context_counts = Counter()
    def train(self, corpus):
        for sentence in corpus:
            tokens = ['<s>'] + sentence.split() +['</s>']
            for i in range(len(tokens) - self.n+1):
                ngram = tuple(tokens[i : i+self.n])
                context = tuple(tokens[i : i+self.n-1])
                self.ngram_counts[ngram] += 1
                self.context_counts[context] +=1
    def get_probability(self, context, target):
        context_tupe = tuple(context)
        ngram_tuple = tuple(context) + (target,)

        numerator = self.ngram_counts.get(ngram_tuple,0)
        denoinator = self.context_counts.get(context_tupe,0)
        if denoinator == 0:
            return 0.0
        return numerator / denoinator

    

In [2]:
train_corpus = [
    '나는 오늘 파이토치 학습을한다','나는 내일 자연어 처리를 한다','나는 오늘 머신러닝 논문을 읽는다'
]
bigram_model = MLELanguateModel(n=2)
bigram_model.train(train_corpus)

In [4]:
bigram_model.ngram_counts, bigram_model.context_counts

(Counter({('<s>', '나는'): 3,
          ('나는', '오늘'): 2,
          ('오늘', '파이토치'): 1,
          ('파이토치', '학습을한다'): 1,
          ('학습을한다', '</s>'): 1,
          ('나는', '내일'): 1,
          ('내일', '자연어'): 1,
          ('자연어', '처리를'): 1,
          ('처리를', '한다'): 1,
          ('한다', '</s>'): 1,
          ('오늘', '머신러닝'): 1,
          ('머신러닝', '논문을'): 1,
          ('논문을', '읽는다'): 1,
          ('읽는다', '</s>'): 1}),
 Counter({('<s>',): 3,
          ('나는',): 3,
          ('오늘',): 2,
          ('파이토치',): 1,
          ('학습을한다',): 1,
          ('내일',): 1,
          ('자연어',): 1,
          ('처리를',): 1,
          ('한다',): 1,
          ('머신러닝',): 1,
          ('논문을',): 1,
          ('읽는다',): 1}))

In [6]:
# 나는 오늘  논리적 확률 2/3
bigram_model.get_probability(['나는'], '오늘'), 2/3

(0.6666666666666666, 0.6666666666666666)